# Train A2C on continuous actions

A2C learns from multi-step rollouts and generalized advantage estimates. This notebook trains it on `Pendulum-v1` with a squashed diagonal-Gaussian actor for bounded continuous actions; its rollout, advantage, and value-target calculations remain the same as in the discrete-action version.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import A2C, A2CConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "Pendulum-v1"

In [ ]:
env = gym.make(ENV_ID)
config = A2CConfig(
    learning_rate=3e-4,
    value_learning_rate=1e-3,
    gamma=0.99,
    n_steps=64,
    gae_lambda=0.95,
    normalize_advantage=True,
    entropy_coefficient=1e-3,
    seed=7,
)

agent = A2C(env, config=config, device="cpu")
agent.learn(total_timesteps=50_000)
env.close()

In [ ]:
window = 10
returns = np.asarray(agent.episode_returns)
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.plot(np.arange(window, len(returns) + 1), moving_average)
plt.xlabel("Episode")
plt.ylabel(f"Mean return ({window} episodes)")
plt.title("A2C on Pendulum-v1")
plt.show()

In [ ]:
eval_env = gym.make(ENV_ID)
result = evaluate_policy(agent, eval_env, episodes=10, deterministic=True)
eval_env.close()

print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")